In [ ]:
# -*- coding: utf-8 -*-
# PyMC / pytensor のコンパイルフラグは import より前に設定する必要がある
# mode=FAST_RUN は fork 後にキャッシュロック競合を引き起こすため除外
import os
os.environ.setdefault("PYTENSOR_FLAGS", "floatX=float64,allow_gc=False")

import concurrent.futures
import copy
import logging
import multiprocessing
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import pymc as pm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

logging.getLogger('pymc').setLevel(logging.ERROR)
logging.getLogger('pytensor').setLevel(logging.ERROR)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')


def find_project_root() -> Path:
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'data').is_dir():
            return p
    raise FileNotFoundError('data/ ディレクトリが見つかりません。リポジトリルートから実行してください。')


ROOT        = find_project_root()
EXP022_DIR  = ROOT / 'EXP022'
MODEL_DIR   = EXP022_DIR / 'models'
RESULTS_DIR = EXP022_DIR / 'results'

print(f'Device      : {device}')
print(f'Project root: {ROOT}')
print(f'Model dir   : {MODEL_DIR}')
print(f'Results dir : {RESULTS_DIR}')

In [4]:
@dataclass
class Config:
    # Network
    # input_size=2: [theta_hat, posterior_variance]
    input_size:    int   = 2
    first_hidden:  int   = 50
    second_hidden: int   = 30
    dropout_rate:  float = 0.0

    # Training
    test_length:         int   = 40
    gamma:               float = 0.1
    memory_capacity:     int   = 1000
    epsilon:             float = 0.1
    batch_size:          int   = 128
    q_network_iteration: int   = 40
    learning_rate:       float = 1e-3
    training_size:       int   = 1000
    validation_size:     int   = 200
    validation_interval: int   = 50

    # PyMC ADVI（Mac 向けに削減済み）
    n_advi_iter:    int   = 500   # EXP018 の 2000 から削減。1次元問題には十分
    n_advi_samples: int   = 200   # EXP018 の 500 から削減
    prior_std:      float = 1.0

    # 並列数: -1=全コア, 1=逐次（デバッグ用）
    n_jobs: int = -1

    # Bank / prior
    bank_type: str = 'uncor'   # 'uncor' | 'cor'
    bank_id:   int = 1
    prior:     str = 'normal'
    n_items:   int = 500

In [ ]:
def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    return (np.random.random(size=p.shape) <= p).astype(int)


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2 * a**2 * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def posterior_advi(
    item_paras: np.ndarray,
    resp: np.ndarray,
    n_iter: int = 500,
    n_samples: int = 200,
    prior_std: float = 1.0,
    D: float = 1.0,
) -> Tuple[float, float]:
    """PyMC 平均場 ADVI で事後平均・事後分散を返す。"""
    a = item_paras[:, 0].astype(float)
    b = item_paras[:, 1].astype(float)
    c = item_paras[:, 2].astype(float)
    r = resp.astype(float)

    with pm.Model():
        theta = pm.Normal('theta', mu=0.0, sigma=prior_std)
        p_raw = c + (1.0 - c) / (1.0 + pm.math.exp(-D * a * (theta - b)))
        p_obs = pm.math.clip(p_raw, 1e-6, 1.0 - 1e-6)
        pm.Bernoulli('x', p=p_obs, observed=r)
        approx = pm.fit(n=n_iter, method='advi', progressbar=False)
        idata  = approx.sample(n_samples)

    samples = idata.posterior['theta'].values.flatten()
    return float(samples.mean()), float(samples.var())


def _init_worker():
    """各ワーカープロセスに独立した pytensor コンパイルキャッシュを割り当てる。
    fork 後のキャッシュロック競合を防ぐ。"""
    import pytensor
    pytensor.config.base_compiledir = tempfile.mkdtemp(prefix='pytensor_worker_')


def _advi_worker(args: tuple) -> Tuple[float, float]:
    item_paras, resp, n_iter, n_samples, prior_std = args
    return posterior_advi(item_paras, resp, n_iter, n_samples, prior_std)


def parallel_advi(
    item_bank: np.ndarray,
    item_id_mat: np.ndarray,
    resp_mat: np.ndarray,
    n_subjects: int,
    n_iter: int,
    n_samples: int,
    prior_std: float,
    n_jobs: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """受検者ループを並列化して (theta_hat, post_var) の配列を返す。"""
    args = [
        (item_bank[item_id_mat[:, s]], resp_mat[:, s], n_iter, n_samples, prior_std)
        for s in range(n_subjects)
    ]
    if n_jobs == 1:
        results = [_advi_worker(a) for a in args]
    else:
        n_workers = os.cpu_count() if n_jobs == -1 else n_jobs
        ctx = multiprocessing.get_context('fork')
        with concurrent.futures.ProcessPoolExecutor(
            max_workers=n_workers,
            mp_context=ctx,
            initializer=_init_worker,
        ) as executor:
            results = list(executor.map(_advi_worker, args))

    theta_arr = np.array([r[0] for r in results])
    var_arr   = np.array([r[1] for r in results])
    return theta_arr, var_arr


def Apply_Positive_Constraint(model, min_value=0.0):
    for param in model.parameters():
        param.data = torch.clamp(param.data, min=min_value)

In [6]:
class Net(nn.Module):
    def __init__(self, input_size, first_hidden, second_hidden, action_space, dropout_rate):
        super().__init__()
        self.fc1     = nn.Linear(input_size, first_hidden)
        self.fc2     = nn.Linear(first_hidden, second_hidden)
        self.out     = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.fc1(x))
        x = F.relu(x)
        x = self.dropout(self.fc2(x))
        x = F.relu(x)
        return self.out(x)

    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)

In [7]:
def Choose_Action(item_id, state, epsilon):
    """state: 1-D array shape (2,) = [theta_hat, post_var]"""
    if np.random.rand() >= epsilon:
        state_t      = torch.unsqueeze(torch.FloatTensor(state), 0).to(device)
        item_id_t    = torch.from_numpy(item_id).to(device).long()
        action_value = eval_net(state_t)
        action_value[:, item_id_t] = torch.zeros(item_id_t.shape).to(device)
        action = torch.max(action_value, -1)[1].cpu().numpy()
    else:
        if any(item_id):
            action = np.random.choice(np.delete(np.arange(action_space), item_id)).astype('int64').reshape(1,)
        else:
            action = np.random.choice(np.arange(action_space)).astype('int64').reshape(1,)
    return action


def Choose_Action_Test(item_id, state):
    """state: 2-D array shape (2, n_subjects) = [[theta_hats], [post_vars]]"""
    state_t      = torch.FloatTensor(state.swapaxes(0, 1)).to(device)
    action_value = eval_net(state_t).detach().cpu().numpy()
    if item_id.shape[0] > 0:
        action_value[
            np.tile(np.arange(item_id.shape[1])[np.newaxis, :], (item_id.shape[0], 1)),
            item_id,
        ] = np.zeros(item_id.shape)
    return action_value.argmax(axis=1)

In [8]:
def TRAIN(cfg):
    best_valid = None
    best_state = None

    loss_func  = nn.MSELoss()
    eval_net.train()
    optimizer  = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)

    # memory layout: [state(2), action(1), reward(1), next_state(2)] = 6 cols
    memory             = np.zeros((cfg.memory_capacity, cfg.input_size * 2 + 2))
    memory_counter     = 0
    learn_step_counter = 0

    if cfg.prior == 'normal':
        training_theta = np.random.randn(cfg.training_size)
    else:
        training_theta = np.random.uniform(-3, 3, cfg.training_size)

    for j in range(cfg.training_size):
        # 訓練は受検者を 1 人ずつ逐次処理（ADVI 1 回/ステップ）
        state   = np.array([np.random.rand() - 0.5, cfg.prior_std ** 2])
        item_id = np.array([]).astype('int64')
        resp    = np.array([]).astype('int64')

        for i in range(cfg.test_length):
            action  = Choose_Action(item_id, state, cfg.epsilon)
            item_id = np.concatenate((item_id, action))
            resp    = np.concatenate((resp, RESPOND(item_bank[action], training_theta[j])))
            reward  = FI(item_bank[action,], training_theta[j])

            post_mean, post_var = posterior_advi(
                item_bank[item_id], resp,
                n_iter=cfg.n_advi_iter,
                n_samples=cfg.n_advi_samples,
                prior_std=cfg.prior_std,
            )
            next_state = np.array([post_mean, post_var])

            memory[memory_counter % cfg.memory_capacity, :] = np.hstack(
                (state, action, reward, next_state)
            )
            memory_counter += 1
            state = next_state

            if memory_counter >= cfg.batch_size:
                batch_memory     = memory[np.random.choice(min(memory_counter, cfg.memory_capacity), cfg.batch_size), :]
                batch_state      = torch.FloatTensor(batch_memory[:, :cfg.input_size]).to(device)
                batch_action     = torch.LongTensor(batch_memory[:, cfg.input_size:cfg.input_size + 1].astype(int)).to(device)
                batch_reward     = torch.FloatTensor(batch_memory[:, cfg.input_size + 1:cfg.input_size + 2]).to(device)
                batch_next_state = torch.FloatTensor(batch_memory[:, -cfg.input_size:]).to(device)

                q_eval   = eval_net(batch_state).gather(1, batch_action)
                q_next   = target_net(batch_next_state).detach()
                q_target = (
                    batch_reward
                    if i == cfg.test_length - 1
                    else batch_reward + cfg.gamma * q_next.max(1)[0].view(cfg.batch_size, 1)
                )
                loss = loss_func(q_eval, q_target)

                optimizer.zero_grad()
                loss.backward()
                Apply_Positive_Constraint(eval_net)
                optimizer.step()

                learn_step_counter += 1
                if learn_step_counter % cfg.q_network_iteration == 0:
                    target_net.load_state_dict(eval_net.state_dict())

        ### Validation（受検者を並列化）###
        if (j + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_theta = np.random.choice(training_theta, cfg.validation_size)
            valid_bias  = np.zeros((cfg.test_length, cfg.validation_size))

            state = np.vstack([
                np.random.rand(cfg.validation_size) - 0.5,
                np.full(cfg.validation_size, cfg.prior_std ** 2),
            ])
            item_id = np.array([])

            for i in range(cfg.test_length):
                action = Choose_Action_Test(item_id, state)
                if i == 0:
                    item_id = action[np.newaxis, :]
                    resp    = RESPOND(item_bank[action,], valid_theta)[np.newaxis, :]
                else:
                    item_id = np.concatenate((item_id, action[np.newaxis, :]))
                    resp    = np.concatenate((resp, RESPOND(item_bank[action,], valid_theta)[np.newaxis, :]))

                theta_0, var_0 = parallel_advi(
                    item_bank, item_id, resp, cfg.validation_size,
                    cfg.n_advi_iter, cfg.n_advi_samples, cfg.prior_std, cfg.n_jobs,
                )
                state = np.vstack([theta_0, var_0])
                valid_bias[i] = theta_0 - valid_theta

            step_valid = np.transpose(np.vstack((
                np.arange(1, cfg.test_length + 1),
                np.mean(valid_bias, axis=1),
                np.sqrt(np.mean(valid_bias ** 2, axis=1)),
                np.mean(abs(valid_bias), axis=1),
            )))
            print('subject: {}\n\n{}\n'.format(j + 1, step_valid))

            result_valid = np.mean(step_valid[6:, 1:], axis=0)
            if best_valid is None or result_valid[1] < best_valid[1]:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())

            eval_net.train()

    return best_state

In [9]:
def TEST(cfg, theta_test):
    with torch.no_grad():
        eval_net.eval()
        testing_size = len(theta_test)

        state = np.vstack([
            np.random.rand(testing_size) - 0.5,
            np.full(testing_size, cfg.prior_std ** 2),
        ])
        item_id  = np.array([])
        dqn_step = np.zeros((1, 4))

        for i in range(cfg.test_length):
            action = Choose_Action_Test(item_id, state)
            if i == 0:
                item_id = action[np.newaxis, :]
                resp    = RESPOND(item_bank[action,], theta_test)[np.newaxis, :]
            else:
                item_id = np.concatenate((item_id, action[np.newaxis, :]))
                resp    = np.concatenate((resp, RESPOND(item_bank[action,], theta_test)[np.newaxis, :]))

            # テストも受検者を並列化
            theta_0, var_0 = parallel_advi(
                item_bank, item_id, resp, testing_size,
                cfg.n_advi_iter, cfg.n_advi_samples, cfg.prior_std, cfg.n_jobs,
            )

            if i == 0:
                theta = theta_0[np.newaxis, :]
            else:
                theta = np.concatenate((theta, theta_0[np.newaxis, :]))

            dqn_step = np.vstack([dqn_step, np.array([
                i + 1,
                np.mean(theta_0 - theta_test),
                np.sqrt(np.mean((theta_0 - theta_test) ** 2)),
                np.mean(abs(theta_0 - theta_test)),
            ])])
            print('step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}'.format(*dqn_step[-1]))

            state = np.vstack([theta_0, var_0])

        user_id_col   = np.repeat(np.arange(1, testing_size + 1), cfg.test_length).reshape(-1, 1)
        step_col      = np.tile(np.arange(1, cfg.test_length + 1), testing_size).reshape(-1, 1)
        item_id_col   = (item_id + 1).transpose().reshape(-1, 1)
        resp_col      = resp.transpose().reshape(-1, 1)
        theta_est_col = theta.transpose().reshape(-1, 1)
        bias_col      = (theta - theta_test).transpose().reshape(-1, 1)

        dqn_data = pd.DataFrame(
            np.hstack([user_id_col, step_col, item_id_col, resp_col, theta_est_col, bias_col]),
            columns=['userID', 'step', 'itemID', 'resp', 'theta_est', 'bias'],
        )
        dqn_summary = pd.DataFrame(dqn_step[1:], columns=['step', 'Bias', 'RMSE', 'MAE'])

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        stem = f"{cfg.bank_type}_{cfg.bank_id}_DQN_{cfg.prior}_gamma_{cfg.gamma}"
        dqn_data.to_csv(   RESULTS_DIR / f'records_{stem}.csv', index=False)
        dqn_summary.to_csv(RESULTS_DIR / f'summary_{stem}.csv', index=False)
        print(f'\nSaved to {RESULTS_DIR}')

In [10]:
cfg = Config(
    # Network
    input_size    = 2,
    first_hidden  = 50,
    second_hidden = 30,
    dropout_rate  = 0.0,

    # Training
    test_length         = 40,
    gamma               = 0.1,
    memory_capacity     = 1000,
    epsilon             = 0.1,
    batch_size          = 128,
    q_network_iteration = 40,
    learning_rate       = 1e-3,
    training_size       = 1000,
    validation_size     = 200,
    validation_interval = 50,

    # PyMC ADVI
    n_advi_iter    = 500,
    n_advi_samples = 200,
    prior_std      = 1.0,

    # 並列数（-1=全コア, 1=逐次）
    n_jobs = -1,

    # Bank
    bank_type = 'uncor',
    bank_id   = 1,
    prior     = 'normal',
    n_items   = 500,
)

bank_dir  = ROOT / 'data' / 'uncorrelated_banks'
item_bank = np.array(
    pd.read_csv(bank_dir / f'item_bank_{cfg.bank_type}_{cfg.bank_id}.csv')[['a', 'b', 'c']]
)[:cfg.n_items]
action_space = item_bank.shape[0]

theta_test = np.array(
    pd.read_csv(ROOT / 'data' / 'theta_true' / f'theta_true_{cfg.bank_id}.csv')['x']
)

print(f'item bank  : {item_bank.shape}')
print(f'theta_test : {theta_test.shape}')
print(f'n_jobs     : {cfg.n_jobs}  ({os.cpu_count()} logical cores)')
print(f'\nConfig:\n{cfg}')

item bank  : (500, 3)
theta_test : (5000,)
n_jobs     : -1  (10 logical cores)

Config:
Config(input_size=2, first_hidden=50, second_hidden=30, dropout_rate=0.0, test_length=40, gamma=0.1, memory_capacity=1000, epsilon=0.1, batch_size=128, q_network_iteration=40, learning_rate=0.001, training_size=1000, validation_size=200, validation_interval=50, n_advi_iter=500, n_advi_samples=200, prior_std=1.0, n_jobs=-1, bank_type='uncor', bank_id=1, prior='normal', n_items=500)


In [11]:
eval_net   = Net(cfg.input_size, cfg.first_hidden, cfg.second_hidden, action_space, cfg.dropout_rate).to(device)
target_net = Net(cfg.input_size, cfg.first_hidden, cfg.second_hidden, action_space, cfg.dropout_rate).to(device)
eval_net.initialize()
target_net.initialize()

best_state = TRAIN(cfg)

assert best_state is not None, 'チェックポイントが保存されませんでした。training_size を増やすか validation_interval を下げてください。'
eval_net.load_state_dict(best_state)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / f"dqn_{cfg.prior}_{cfg.bank_type}_{cfg.bank_id}_gamma_{cfg.gamma}.pt"
torch.save(eval_net.state_dict(), model_path)
print(f'Model saved to: {model_path}')

TEST(cfg, theta_test)

Timeout: The file lock 'The file lock '/Users/itsuki/.pytensor/compiledir_macOS-15.5-arm64-arm-64bit-arm-3.11.8-64/.lock' could not be acquired.
Apply node that caused the error: Composite{...}([0.2547238 ... .23405096], [0.7452761 ... .76594904], [-1.957613 ... .83091746], DimShuffle{order=[x]}.0, [-0.126309 ... .31283109], x{[1 0 1]}, [1.4589623 ... .40238947])
Toposort index: 21
Inputs types: [TensorType(float64, shape=(3,)), TensorType(float64, shape=(3,)), TensorType(float64, shape=(3,)), TensorType(float64, shape=(1,)), TensorType(float64, shape=(3,)), TensorType(int64, shape=(3,)), TensorType(float64, shape=(3,))]

HINT: Use a linker other than the C linker to print the inputs' shapes and strides.
HINT: Re-running with most PyTensor optimizations disabled could provide a back-trace showing when this node was created. This can be done by setting the PyTensor flag 'optimizer=fast_compile'. If that does not work, PyTensor optimizations can be disabled with 'optimizer=None'.
HINT: U